# Fuente 2 - API Segmento Vial de Medellín

## Objetivo

Consultar la capa Segmento Vial - Flujo de la Secretaría de Movilidad de Medellín para conocer su estructura, identificar las variables disponibles y verificar que contiene información geográfica que permita localizar y caracterizar el segmento vial correspondiente a la Avenida Las Palmas y, posteriormente, al sector de La Asomadera #2.

En esta etapa se busca conocer la información disponible y comprobar que la fuente es adecuada para representar espacialmente el tramo de estudio.

## 1. Identificación de la fuente

**Nombre:** Segmento Vial - Flujo  
**Entidad:** Secretaría de Movilidad de Medellín  
**Tipo de fuente:** API REST de ArcGIS  
**Tipo de información:** Segmentos de la malla vial utilizados para el manejo de información asociada a movilidad.

Primero se consultarán los metadatos de la fuente para conocer su estructura y las variables disponibles.
fuente: https://www.medellin.gov.co/servidormapas/rest/services/transporte/VA_SIGMOV_Segmentovial/MapServer/1


In [ ]:
import requests
import pandas as pd

# URL de la capa Segmento Vial - Flujo
url_capa = "https://www.medellin.gov.co/servidormapas/rest/services/transporte/VA_SIGMOV_Segmentovial/MapServer/1"

respuesta = requests.get(
    url_capa,
    params={"f": "json"},
    timeout=30
)

respuesta.raise_for_status()

metadata = respuesta.json()

print("Conexión realizada correctamente")
print("Nombre de la capa:", metadata.get("name"))
print("Tipo de capa:", metadata.get("type"))
print("Tipo de geometría:", metadata.get("geometryType"))

Conexión realizada correctamente
Nombre de la capa: Segmento Vial - Flujo
Tipo de capa: Feature Layer
Tipo de geometría: esriGeometryPolyline


## 2. Variables disponibles en la fuente

Se consultan los campos disponibles en la API para conocer qué información describe cada segmento vial e identificar posteriormente cuáles variables son útiles para el proyecto.

In [ ]:
campos = metadata.get("fields", [])

tabla_campos = pd.DataFrame([
    {
        "nombre_variable": campo.get("name"),
        "descripcion": campo.get("alias"),
        "tipo_arcgis": campo.get("type")
    }
    for campo in campos
])

tabla_campos

,nombre_variable,descripcion,tipo_arcgis
0,objectid,OBJECTID,esriFieldTypeOID
1,id_svial,id_svial,esriFieldTypeInteger
2,nombre,NOMBRE,esriFieldTypeString
3,jerarquia_op,JERARQUIA_OP,esriFieldTypeSmallInteger
4,oneway,ONEWAY,esriFieldTypeString
5,nro_carriles,NRO_CARRILES,esriFieldTypeSmallInteger
6,paso_nivel,PASO_NIVEL,esriFieldTypeSmallInteger
7,f_elev,F_ELEV,esriFieldTypeSmallInteger
8,t_elev,T_ELEV,esriFieldTypeSmallInteger
9,actualizado,ACTUALIZO,esriFieldTypeSmallInteger


## 3. Consulta de segmentos asociados a Las Palmas

Después de conocer las variables disponibles, se realiza una búsqueda sobre el campo `nombre` para identificar cómo están registrados en la fuente los segmentos relacionados con la Avenida Las Palmas.

Se realiza una búsqueda utilizando la forma "Palmas".

In [ ]:
url_consulta = f"{url_capa}/query"

parametros = {
    "where": "nombre LIKE '%Palmas%'",
    "outFields": "id_svial,nombre,jerarquia_op,oneway,nro_carriles,municipio,st_length(shape)",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "json"
}

respuesta_palmas = requests.get(
    url_consulta,
    params=parametros,
    timeout=30
)

respuesta_palmas.raise_for_status()

datos_palmas = respuesta_palmas.json()

if "error" in datos_palmas:
    print("La API devolvió un error:")
    print(datos_palmas["error"])
else:
    print("Consulta realizada correctamente")
    print("Segmentos encontrados:", len(datos_palmas.get("features", [])))

Consulta realizada correctamente
Segmentos encontrados: 134


In [ ]:
registros_palmas = [
    feature["attributes"]
    for feature in datos_palmas["features"]
]

df_palmas = pd.DataFrame(registros_palmas)

df_palmas.head(20)

,id_svial,nombre,jerarquia_op,oneway,nro_carriles,municipio,st_length(shape)
0,105145,Av. Las Palmas >Oriente,2.0,FT,NaN,Envigado,26.188081
1,110965,Av. Las Palmas >Occidente,2.0,FT,NaN,Envigado,350.848738
2,110755,Av. Las Palmas >Occidente,2.0,FT,NaN,Medellin,465.624555
3,106931,Via Las Palmas -Aeropuerto JMC,2.0,DW,NaN,Envigado,341.862322
4,106932,Via Las Palmas -Aeropuerto JMC,2.0,DW,NaN,Envigado,605.638442
5,106943,Via Las Palmas -Aeropuerto JMC,2.0,DW,NaN,Envigado,161.140700
6,117201,Av. Las Palmas >Oriente,2.0,FT,NaN,Medellin,421.040542
7,115970,Av. Las Palmas >Occidente,2.0,FT,NaN,Medellin,274.772211
8,117303,Av. Las Palmas >Occidente,2.0,FT,NaN,Medellin,137.330403
9,115927,Av. Las Palmas >Occidente,2.0,FT,NaN,Medellin,11.931961


### Selección de segmentos dentro de Medellín

La búsqueda inicial encontró segmentos asociados a la palabra "Palmas" tanto en Medellín como en otros municipios y vías relacionadas.

Como el proyecto se desarrolla en el sector de La Asomadera #2, dentro de Medellín, se seleccionan únicamente los registros correspondientes al municipio de Medellín cuyo nombre contiene "Av. Las Palmas".

Esta selección permite reducir el conjunto de segmentos candidatos antes de identificar espacialmente cuál corresponde al tramo específico de estudio.

In [ ]:
df_palmas_medellin = df_palmas[
    (df_palmas["municipio"] == "Medellin") &
    (df_palmas["nombre"].str.contains("Av. Las Palmas", case=False, na=False))
].copy()

print("Segmentos de Av. Las Palmas encontrados en Medellín:",
      len(df_palmas_medellin))

df_palmas_medellin

Segmentos de Av. Las Palmas encontrados en Medellín: 88


,id_svial,nombre,jerarquia_op,oneway,nro_carriles,municipio,st_length(shape)
2,110755,Av. Las Palmas >Occidente,2.0,FT,NaN,Medellin,465.624555
6,117201,Av. Las Palmas >Oriente,2.0,FT,NaN,Medellin,421.040542
7,115970,Av. Las Palmas >Occidente,2.0,FT,NaN,Medellin,274.772211
8,117303,Av. Las Palmas >Occidente,2.0,FT,NaN,Medellin,137.330403
9,115927,Av. Las Palmas >Occidente,2.0,FT,NaN,Medellin,11.931961
...,...,...,...,...,...,...,...
128,138406,Av. Las Palmas > Oriente - Calle 19,NaN,FT,2.0,Medellin,114.694773
129,138388,Av. Las Palmas > Occidente - Calle 19,NaN,FT,2.0,Medellin,126.993994
130,138387,Av. Las Palmas > Occidente - Calle 19,NaN,FT,1.0,Medellin,141.438176
131,113979,Av. Las Palmas >Oriente - Calle 19,2.0,FT,2.0,Medellin,117.829121


## 4. Información geográfica de los segmentos

La capa representa cada registro mediante una geometría de tipo línea (`Polyline`). Esta información permitirá ubicar espacialmente los diferentes segmentos de la Avenida Las Palmas.

En la consulta anterior se identificaron 88 segmentos asociados a la Avenida Las Palmas dentro de Medellín. En esta etapa no se determinará todavía cuál corresponde específicamente a La Asomadera #2; esta selección se realizará posteriormente cuando se extraigan los datos específicos del área de estudio.

A continuación se observa, únicamente como verificación, la geometría de uno de los segmentos consultados.

In [ ]:
# Buscamos la geometría del primer segmento de Av. Las Palmas ubicado en Medellín

ids_medellin = set(df_palmas_medellin["id_svial"])

segmento_ejemplo = next(
    feature for feature in datos_palmas["features"]
    if feature["attributes"]["id_svial"] in ids_medellin
)

print("ID del segmento:", segmento_ejemplo["attributes"]["id_svial"])
print("Nombre:", segmento_ejemplo["attributes"]["nombre"])
print("Geometría:")
print(segmento_ejemplo["geometry"])

ID del segmento: 110755
Nombre: Av. Las Palmas >Occidente
Geometría:
{'paths': [[[-75.54715810048468, 6.195280899605096], [-75.54709130049044, 6.195396399456934], [-75.54702439985007, 6.195494501129407], [-75.54694140075522, 6.195607899912169], [-75.54688220005501, 6.195726400162282], [-75.54682050031579, 6.195860300352945], [-75.54677220057351, 6.196064600597878], [-75.54676010077563, 6.196211200295301], [-75.54675930008392, 6.196321499986205], [-75.54678940037431, 6.196473799850594], [-75.54684720074333, 6.196598199620536], [-75.54692720055819, 6.19678870057412], [-75.54700939996019, 6.197098700511581], [-75.54708740042355, 6.197471000088978], [-75.54710770061402, 6.19764169946506], [-75.54710769960123, 6.197923300329096], [-75.54704760036924, 6.198332899910027], [-75.54698750012729, 6.198503600593957], [-75.54690170013376, 6.198648599859447], [-75.5464707004185, 6.1991380999681756]]]}


## Conclusión de la exploración

La consulta confirma que la capa Segmento Vial - Flujo puede utilizarse como fuente geográfica del proyecto. La API contiene segmentos asociados a la Avenida Las Palmas dentro de Medellín y proporciona un identificador único (`id_svial`), características generales del segmento y su geometría lineal.

La búsqueda realizada encontró 88 segmentos asociados a la Avenida Las Palmas en Medellín, lo que evidencia que la vía está dividida en múltiples segmentos independientes. Para el proyecto será necesario identificar posteriormente cuál de estos segmentos, o qué conjunto de ellos, corresponde específicamente al sector de La Asomadera #2.

En esta etapa únicamente se verificó el acceso, la estructura y la disponibilidad de la información geográfica de la fuente.

## 5. Adquisición de los segmentos de Avenida Las Palmas

Después de comprobar que la fuente contiene los segmentos de la Avenida Las Palmas, se realiza una consulta para obtener los segmentos ubicados dentro del municipio de Medellín.

Estos datos se conservarán sin realizar todavía la selección del tramo específico de La Asomadera #2 ni cálculos derivados de su geometría. Esa preparación se realizará posteriormente.

Debido a que la geometría lineal es una parte fundamental de esta fuente, los datos se almacenarán en formato GeoJSON, el cual permite conservar tanto los atributos como las coordenadas de cada segmento.

In [ ]:
parametros_geojson = {
    "where": "nombre LIKE '%Av. Las Palmas%' AND municipio = 'Medellin'",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

respuesta_geojson = requests.get(
    url_consulta,
    params=parametros_geojson,
    timeout=60
)

respuesta_geojson.raise_for_status()

datos_geojson = respuesta_geojson.json()

print("Consulta realizada correctamente")
print(
    "Segmentos obtenidos:",
    len(datos_geojson.get("features", []))
)

Consulta realizada correctamente
Segmentos obtenidos: 87


### Almacenamiento de la fuente adquirida

La respuesta geográfica obtenida desde la API se almacena en formato GeoJSON. No se realizan modificaciones sobre los atributos ni sobre la geometría en esta etapa.

In [ ]:
import json

nombre_archivo = "segmentos_las_palmas_medellin.geojson"

with open(
    nombre_archivo,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        datos_geojson,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print("Archivo generado:", nombre_archivo)
print(
    "Segmentos almacenados:",
    len(datos_geojson.get("features", []))
)

Archivo generado: segmentos_las_palmas_medellin.geojson
Segmentos almacenados: 87


###DESCARGAR

In [ ]:
from google.colab import files

files.download("segmentos_las_palmas_medellin.geojson")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Calidad de los datos

Se revisan todos los atributos almacenados en el GeoJSON para identificar datos faltantes y errores básicos de consistencia.

También se presenta una segunda revisión únicamente con las variables relevantes para el proyecto.

In [ ]:
import pandas as pd
import json
from IPython.display import display

# =========================================================
# 6. CALIDAD DE LOS DATOS - SEGMENTO VIAL
# =========================================================

# Leemos directamente el GeoJSON generado
with open(
    "segmentos_las_palmas_medellin.geojson",
    "r",
    encoding="utf-8"
) as archivo:

    geojson = json.load(archivo)


# Convertimos temporalmente las propiedades a DataFrame
registros = []

for feature in geojson.get("features", []):

    fila = dict(
        feature.get("properties") or {}
    )

    fila["geometry"] = feature.get("geometry")

    registros.append(fila)


df = pd.DataFrame(registros)

print("Registros analizados:", len(df))
print("Variables analizadas:", len(df.columns))


# ---------------------------------------------------------
# 1. Datos faltantes
# ---------------------------------------------------------

marcadores_sin_informacion = {
    "ND",
    "N/D",
    "SIN INFORMACION",
    "SIN INFORMACIÓN",
    "NO DISPONIBLE",
    "NULL"
}


def analizar_ausencia(serie):

    faltante = serie.isna().copy()

    codificado = pd.Series(
        False,
        index=serie.index
    )

    if (
        pd.api.types.is_object_dtype(serie)
        or pd.api.types.is_string_dtype(serie)
    ):

        texto = (
            serie.astype("string")
            .str.strip()
        )

        faltante = (
            faltante
            | texto.eq("").fillna(False)
        )

        codificado = (
            texto.str.upper()
            .isin(marcadores_sin_informacion)
            .fillna(False)
        )

    return faltante, codificado


faltantes = {}
faltantes_codificados = {}

for columna in df.columns:

    mascara_faltante, mascara_codificada = (
        analizar_ausencia(df[columna])
    )

    faltantes[columna] = mascara_faltante
    faltantes_codificados[columna] = mascara_codificada


# ---------------------------------------------------------
# 2. Errores y criterios
# ---------------------------------------------------------

errores = {
    columna: pd.Series(
        False,
        index=df.index
    )
    for columna in df.columns
}

criterios = {
    columna: "Se revisan datos faltantes"
    for columna in df.columns
}


# objectid e id_svial
for columna in ["objectid", "id_svial"]:

    numero = pd.to_numeric(
        df[columna],
        errors="coerce"
    )

    errores[columna] = (
        ~faltantes[columna]
        & (
            numero.isna()
            | (numero <= 0)
            | df[columna].duplicated(keep=False)
        )
    )

    criterios[columna] = (
        "Numérico, positivo y no duplicado"
    )


# nombre
errores["nombre"] = (
    ~faltantes["nombre"]
    & ~df["nombre"]
    .astype("string")
    .str.contains(
        "Las Palmas",
        case=False,
        na=False
    )
)

criterios["nombre"] = (
    "Debe corresponder a Las Palmas"
)


# Variables que deben ser numéricas cuando existen
for columna in [
    "jerarquia_op",
    "paso_nivel",
    "f_elev",
    "t_elev",
    "actualizado"
]:

    numero = pd.to_numeric(
        df[columna],
        errors="coerce"
    )

    errores[columna] = (
        ~faltantes[columna]
        & numero.isna()
    )

    criterios[columna] = (
        "Cuando existe, debe ser numérico"
    )


# nro_carriles
carriles = pd.to_numeric(
    df["nro_carriles"],
    errors="coerce"
)

errores["nro_carriles"] = (
    ~faltantes["nro_carriles"]
    & (
        carriles.isna()
        | (carriles <= 0)
        | ((carriles % 1) != 0)
    )
)

criterios["nro_carriles"] = (
    "Cuando existe, debe ser entero mayor que 0"
)


# fecha_act
fecha_numero = pd.to_numeric(
    df["fecha_act"],
    errors="coerce"
)

fecha_convertida = pd.to_datetime(
    fecha_numero,
    unit="ms",
    errors="coerce"
)

errores["fecha_act"] = (
    ~faltantes["fecha_act"]
    & (
        fecha_numero.isna()
        | fecha_convertida.isna()
    )
)

criterios["fecha_act"] = (
    "Cuando existe, debe ser timestamp ArcGIS válido"
)


# municipio
errores["municipio"] = (
    ~faltantes["municipio"]
    & ~df["municipio"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("medellin")
)

criterios["municipio"] = (
    "Debe corresponder a Medellin"
)


# longitud del segmento
longitud_segmento = pd.to_numeric(
    df["st_length(shape)"],
    errors="coerce"
)

errores["st_length(shape)"] = (
    ~faltantes["st_length(shape)"]
    & (
        longitud_segmento.isna()
        | (longitud_segmento <= 0)
    )
)

criterios["st_length(shape)"] = (
    "Numérica y mayor que 0"
)


# geometry
def geometria_valida(geometria):

    return (
        isinstance(geometria, dict)
        and geometria.get("type")
        in {"LineString", "MultiLineString"}
        and bool(geometria.get("coordinates"))
    )


errores["geometry"] = (
    ~faltantes["geometry"]
    & ~df["geometry"].apply(
        geometria_valida
    )
)

criterios["geometry"] = (
    "LineString o MultiLineString con coordenadas"
)


# =========================================================
# 3. TABLA 1 - TODAS LAS VARIABLES
# =========================================================

tabla_calidad_completa = pd.DataFrame({
    "Variable": df.columns,
    "Tipo de dato": df.dtypes.astype(str).values,
    "Datos faltantes": [
        int(faltantes[col].sum())
        for col in df.columns
    ],
    "Faltantes codificados": [
        int(faltantes_codificados[col].sum())
        for col in df.columns
    ],
    "Datos con error": [
        int(errores[col].sum())
        for col in df.columns
    ],
    "Criterio revisado": [
        criterios[col]
        for col in df.columns
    ]
})


print("\nTABLA 1 - CALIDAD DE TODAS LAS VARIABLES")
display(tabla_calidad_completa)


# =========================================================
# 4. TABLA 2 - VARIABLES RELEVANTES
# =========================================================

variables_relevantes = [
    "id_svial",
    "nombre",
    "oneway",
    "nro_carriles",
    "st_length(shape)",
    "geometry"
]


tabla_calidad_relevantes = (
    tabla_calidad_completa[
        tabla_calidad_completa["Variable"]
        .isin(variables_relevantes)
    ]
    .reset_index(drop=True)
)


print("\nTABLA 2 - VARIABLES RELEVANTES PARA EL PROYECTO")
display(tabla_calidad_relevantes)


# =========================================================
# 5. RESUMEN
# =========================================================

ausencias_completas = pd.DataFrame({
    col: (
        faltantes[col]
        | faltantes_codificados[col]
    )
    for col in df.columns
})

errores_completos = pd.DataFrame(errores)


ausencias_relevantes = pd.DataFrame({
    col: (
        faltantes[col]
        | faltantes_codificados[col]
    )
    for col in variables_relevantes
})

errores_relevantes = pd.DataFrame({
    col: errores[col]
    for col in variables_relevantes
})


print("\nRESUMEN GENERAL")

print(
    "Registros con alguna ausencia de información:",
    int(ausencias_completas.any(axis=1).sum())
)

print(
    "Registros con algún error:",
    int(errores_completos.any(axis=1).sum())
)


print("\nRESUMEN VARIABLES RELEVANTES")

print(
    "Registros con alguna ausencia:",
    int(ausencias_relevantes.any(axis=1).sum())
)

print(
    "Registros con algún error:",
    int(errores_relevantes.any(axis=1).sum())
)

Registros analizados: 87
Variables analizadas: 15

TABLA 1 - CALIDAD DE TODAS LAS VARIABLES


,Variable,Tipo de dato,Datos faltantes,Faltantes codificados,Datos con error,Criterio revisado
0,objectid,int64,0,0,0,"Numérico, positivo y no duplicado"
1,id_svial,int64,0,0,0,"Numérico, positivo y no duplicado"
2,nombre,object,0,0,0,Debe corresponder a Las Palmas
3,jerarquia_op,float64,14,0,0,"Cuando existe, debe ser numérico"
4,oneway,object,0,0,0,Se revisan datos faltantes
5,nro_carriles,float64,36,0,0,"Cuando existe, debe ser entero mayor que 0"
6,paso_nivel,float64,63,0,0,"Cuando existe, debe ser numérico"
7,f_elev,int64,0,0,0,"Cuando existe, debe ser numérico"
8,t_elev,int64,0,0,0,"Cuando existe, debe ser numérico"
9,actualizado,float64,69,0,0,"Cuando existe, debe ser numérico"



TABLA 2 - VARIABLES RELEVANTES PARA EL PROYECTO


,Variable,Tipo de dato,Datos faltantes,Faltantes codificados,Datos con error,Criterio revisado
0,id_svial,int64,0,0,0,"Numérico, positivo y no duplicado"
1,nombre,object,0,0,0,Debe corresponder a Las Palmas
2,oneway,object,0,0,0,Se revisan datos faltantes
3,nro_carriles,float64,36,0,0,"Cuando existe, debe ser entero mayor que 0"
4,st_length(shape),float64,0,0,0,Numérica y mayor que 0
5,geometry,object,0,0,0,LineString o MultiLineString con coordenadas



RESUMEN GENERAL
Registros con alguna ausencia de información: 87
Registros con algún error: 0

RESUMEN VARIABLES RELEVANTES
Registros con alguna ausencia: 36
Registros con algún error: 0
